# Bronze Layer — Idempotent ingestion + NULL guard

AutoLoader land JSON từ Volume inbox vào staging `_bronze_raw` (append, giữ checkpoint), sau đó **MERGE batch** dedup theo `_row_hash` vào `cosmetics_events_bronze` (clean) và `cosmetics_events_bronze_quarantine` (NULL / header / malformed).

**Idempotent**: re-upload cùng file = no-op (MERGE insert-only trên `_row_hash`).

> Yêu cầu: `dq_framework.ipynb` cùng thư mục (DQ gate).

In [ ]:
%run ./dq_framework

from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import col, input_file_name, current_timestamp, md5, concat_ws

CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.bronze_cosmetics"
RAW_TABLE = f"{BRONZE_SCHEMA}.cosmetics_events_bronze_raw"
BRONZE_TABLE = f"{BRONZE_SCHEMA}.cosmetics_events_bronze"
QUARANTINE_TABLE = f"{BRONZE_SCHEMA}.cosmetics_events_bronze_quarantine"
INBOX_PATH = "/Volumes/workspace/bronze_cosmetics/inbox/"
CHECKPOINT_PATH = "/Volumes/workspace/bronze_cosmetics/checkpoints/autoloader_bronze"

schema = (StructType()
    .add("event_time", StringType()).add("event_type", StringType())
    .add("product_id", StringType()).add("category_id", StringType())
    .add("category_code", StringType()).add("brand", StringType())
    .add("price", StringType()).add("user_id", StringType())
    .add("user_session", StringType()))

spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_SCHEMA}")
print("Config ready.")

In [ ]:
# AutoLoader: land JSON tu Volume inbox vao staging _bronze_raw (append, availableNow=batch-like).
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "_schema")
    .schema(schema)
    .load(INBOX_PATH)
    .withColumn("_source_file", input_file_name())
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_row_hash", md5(concat_ws("|",
        col("event_time"), col("event_type"), col("product_id"), col("category_id"),
        col("category_code"), col("brand"), col("price"), col("user_id"), col("user_session")))))

query = (raw_stream.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .table(RAW_TABLE))
query.awaitTermination()
print(f"AutoLoader landing xong -> {RAW_TABLE}")

In [ ]:
from pyspark.sql.functions import isnan

raw = spark.table(RAW_TABLE)
# Quarantine: NULL key, header CSV, hoac price khong phai so hop le (null/NaN sau cast).
# price am nhung la so -> van giu o clean bronze; Silver se loc price>0.
raw_tagged = raw.withColumn("_price_d", col("price").cast("double"))
is_invalid = (
    col("user_id").isNull() | (col("user_id") == "") |
    col("product_id").isNull() | (col("product_id") == "") |
    (col("product_id") == "product_id") |          # dong header CSV
    col("_price_d").isNull() | isnan(col("_price_d"))  # price khong cast duoc / NaN
)
valid_df = raw_tagged.filter(~is_invalid).drop("_price_d")
invalid_df = raw_tagged.filter(is_invalid).drop("_price_d")

# Tao bang dich (empty, schema dung) lan dau — idempotent.
valid_df.createOrReplaceTempView("_valid_bronze")
invalid_df.createOrReplaceTempView("_invalid_bronze")
spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} USING DELTA AS SELECT * FROM _valid_bronze WHERE 1=0")
spark.sql(f"CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} USING DELTA AS SELECT * FROM _invalid_bronze WHERE 1=0")

# MERGE insert-only theo _row_hash -> re-land cung file = no-op (idempotent).
spark.sql(f"""
MERGE INTO {BRONZE_TABLE} AS t
USING _valid_bronze AS s
ON t._row_hash = s._row_hash
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"""
MERGE INTO {QUARANTINE_TABLE} AS t
USING _invalid_bronze AS s
ON t._row_hash = s._row_hash
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Bronze MERGE xong. raw={raw.count()}, clean={spark.table(BRONZE_TABLE).count()}, quarantine={spark.table(QUARANTINE_TABLE).count()}")

In [ ]:
# DQ gate: clean bronze khong duoc co null key (chung minh quarantine hoat dong).
run_gate(spark, BRONZE_TABLE, rule_set="bronze")

spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").select("version","timestamp","operation").show(5, truncate=False)